# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
!git clone https://github.com/YomnaImad07/FlyRank-ML-Internship.git

fatal: destination path 'FlyRank-ML-Internship' already exists and is not an empty directory.


In [8]:
import pandas as pd

df = pd.read_csv("/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv")

df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                              bins=[-1, 90, 180, 365, 10000],
                              labels=["<90d", "90-180d", "180-365d", "365d+"])

bucket_table_1 = df.groupby("stale_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
)
print(bucket_table_1)

                  n  decline_rate
stale_bucket                     
<90d          20655      0.512031
90-180d        9171      0.611057
180-365d        169      0.467456
365d+             5      0.600000


/tmp/ipykernel_280/232471486.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_1 = df.groupby("stale_bucket").agg(


In [9]:
df["position_bucket"] = pd.cut(df["avg_position"],
                                 bins=[-1, 0, 10, 20, 1000],
                                 labels=["no_data(0)", "1-10", "11-20", "20+"])

bucket_table_2 = df[df["avg_position"] > 0].groupby("position_bucket").agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
)
print(bucket_table_2)

                     n  mean_ctr
position_bucket                 
no_data(0)           0       NaN
1-10             12983  0.832373
11-20             7273  0.323443
20+               8539  0.211333


/tmp/ipykernel_280/1693149249.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_2 = df[df["avg_position"] > 0].groupby("position_bucket").agg(


**Signal 1 — Staleness (behind the `stale_visible_page` flag):** Verdict: MIXED — decline rate rises from 51.2% (n=20,655, <90d) to 61.1% (n=9,171, 90-180d), which supports the staleness idea. But it then drops to 46.7% for 180-365d (n=169) and rises again to 60.0% for 365d+ (n=5) — both very small samples, likely too noisy to trust. The relationship is not a clean, monotonic trend, so staleness alone is a partial signal at best, not a reliable standalone rule.

**Signal 2 — CTR vs position (behind the `low_ctr_visible_page` flag):** Verdict: CONFIRMED — mean CTR drops sharply and consistently as position worsens: 83.2% at position 1-10 (n=12,983), down to 32.3% at 11-20 (n=7,273), down to 21.1% at 20+ (n=8,539). This is a clear, large-sample, monotonic relationship that strongly supports the logic behind the CTR-fix flag.

**The rule, in plain words:** A page is worth reviewing first if it is stale (not updated in 180+ days), still gets meaningful search visibility (500+ impressions), and shows a declining trend. Given signal 1's mixed result, staleness alone is a weak lever — the rule leans more heavily on visibility and observed decline, using staleness as a secondary filter rather than the primary driver.

**Reason codes:**
- `stale_declining_visible`: meets all three conditions above.
- `monitor_only`: does not meet all three conditions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
declining = (df["trend_direction"] == "down").astype(int)

df["score"] = stale * visible * declining * df["impressions_90d"]
df["reason_code"] = df["score"].apply(lambda x: "stale_declining_visible" if x > 0 else "monitor_only")
df["action"] = df["score"].apply(lambda x: "review_for_refresh" if x > 0 else "monitor")

ranked_queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "trend_direction"]
]

output_dir = "/content/FlyRank-ML-Internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)
ranked_queue.to_csv(f"{output_dir}/baseline_action_score.csv", index=False)

print("Saved:", os.path.exists(f"{output_dir}/baseline_action_score.csv"))
print(ranked_queue.head(20))

Saved: True
                 content_id          client_id  score  \
16751  content_cf56e2e2e282  client_7f2253d7e2  61678   
16514  content_7368877ea310  client_7f2253d7e2  59472   
7021   content_1bfaa38ff26c  client_7f2253d7e2  25715   
21268  content_0a91db491d14  client_7f2253d7e2  13299   
11489  content_5feee3994adb  client_7f2253d7e2   7812   
12045  content_c2d929d83eaa  client_7f2253d7e2   7558   
698    content_b16bd7307b39  client_7f2253d7e2   4590   
5327   content_fe16a55cd13d  client_7f2253d7e2   4556   
26810  content_ecb6215e79fd  client_7f2253d7e2   4429   
20837  content_928af3e22c80  client_7f2253d7e2   1697   
22872  content_e3ff1b093148  client_d029fa3a95   1408   
26840  content_7f116ae1f6f5  client_9400f1b21c    954   
26799  content_77d4d5930e5e  client_7f2253d7e2    828   
7452   content_72496874f806  client_4ec9599fc2    821   
11630  content_6226ee6adc91  client_d029fa3a95    545   
3507   content_074ba6ead17b  client_d029fa3a95    533   
20003  content_6dff

In [11]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = (df["trend_direction"] == "down").astype(int)
p_at_50 = precision_at_k(df["score"], labels, 50)
base_rate = labels.mean()

print("Precision@50:", p_at_50)
print("Base rate (random guessing):", base_rate)

Precision@50: 0.7
Base rate (random guessing): 0.5420666666666667


The ranked queue was written to `work/outputs/baseline_action_score.csv`. Precision@50 is 0.70, compared to a base rate of 0.542 — showing the rule performs meaningfully better than random ranking, correctly flagging declining pages about 70% of the time among its top 50 picks versus the 54.2% you'd expect by chance.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
top_20 = ranked_queue.head(20)
print(top_20)

                 content_id          client_id  score  \
16751  content_cf56e2e2e282  client_7f2253d7e2  61678   
16514  content_7368877ea310  client_7f2253d7e2  59472   
7021   content_1bfaa38ff26c  client_7f2253d7e2  25715   
21268  content_0a91db491d14  client_7f2253d7e2  13299   
11489  content_5feee3994adb  client_7f2253d7e2   7812   
12045  content_c2d929d83eaa  client_7f2253d7e2   7558   
698    content_b16bd7307b39  client_7f2253d7e2   4590   
5327   content_fe16a55cd13d  client_7f2253d7e2   4556   
26810  content_ecb6215e79fd  client_7f2253d7e2   4429   
20837  content_928af3e22c80  client_7f2253d7e2   1697   
22872  content_e3ff1b093148  client_d029fa3a95   1408   
26840  content_7f116ae1f6f5  client_9400f1b21c    954   
26799  content_77d4d5930e5e  client_7f2253d7e2    828   
7452   content_72496874f806  client_4ec9599fc2    821   
11630  content_6226ee6adc91  client_d029fa3a95    545   
3507   content_074ba6ead17b  client_d029fa3a95    533   
20003  content_6dffb8bfe857  cl

1. **content_cf56e2e2e282** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: High (largest impressions, clearly stale at 194 days) | Would be wrong if: recently manually refreshed but the timestamp wasn't updated.

2. **content_7368877ea310** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: High | Would be wrong if: same timestamp-lag issue as above.

3. **content_1bfaa38ff26c** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: High | Would be wrong if: the decline is seasonal rather than structural.

4. **content_0a91db491d14** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium-High | Would be wrong if: a sibling page absorbed this page's traffic (consolidation).

5. **content_5feee3994adb** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium | Would be wrong if: the drop is a short-term SERP feature change, not lasting decline.

6. **content_c2d929d83eaa** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium | Would be wrong if: same consolidation/seasonality risk as #5.

7. **content_b16bd7307b39** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium | Would be wrong if: this is naturally low-volume content and the drop is just noise.

8. **content_fe16a55cd13d** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium | Would be wrong if: same noise risk as #7.

9. **content_ecb6215e79fd** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Medium | Would be wrong if: the topic is seasonal and naturally dips this time of year.

10. **content_928af3e22c80** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low-Medium | Would be wrong if: impressions (1,697) are too low to justify review priority.

11. **content_e3ff1b093148** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low | Would be wrong if: just crossed the 180-day threshold — a borderline, low-confidence case.

12. **content_7f116ae1f6f5** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low-Medium | Would be wrong if: 954 impressions is too low relative to 301 days stale to be worth editorial time.

13. **content_77d4d5930e5e** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low | Would be wrong if: impressions (828) are close to the 500 minimum — weak visibility signal.

14. **content_72496874f806** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low | Would be wrong if: same borderline visibility concern as #13.

15. **content_6226ee6adc91** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low (weak pick) | Would be wrong if: impressions (545) are barely above threshold — a small data shift removes it entirely.

16. **content_074ba6ead17b** — Action: review_for_refresh | Reason: stale_declining_visible | Confidence: Low (weak pick) | Would be wrong if: same near-threshold concern as #15.

17. **content_6dffb8bfe857** — Action: monitor | Reason: monitor_only | Confidence: Medium | Would be wrong if: 104 days already represents real staleness for this content type, meaning it should have been flagged.

18. **content_c42c3746eeff** — Action: monitor | Reason: monitor_only | Confidence: High | Would be wrong if: trend is "up" — correctly excluded, low risk of being wrong.

19. **content_6a21f0fe9d71** — Action: monitor | Reason: monitor_only | Confidence: High | Would be wrong if: trend is "new" with 1 impression — correctly excluded, too new to judge.

20. **content_9fecdbcad47a** — Action: monitor | Reason: monitor_only | Confidence: High | Would be wrong if: trend is "flat" with 2 impressions — correctly excluded, essentially no data.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Rows 15 and 16 in the top 20 (content_6226ee6adc91 and content_074ba6ead17b) are the weakest picks — their impressions_90d (545 and 533) sit barely above the 500-impression threshold, meaning a small data fluctuation could push them out of the queue entirely. Row 11 (content_e3ff1b093148) is also weak, having just crossed the 180-day staleness threshold at 183 days — a borderline case with low confidence.

**Also notable:** Only 16 of 30,000 pages (about 0.05%) meet all three rule conditions (stale AND visible AND declining) at once. The rule is very conservative — a real editorial team might find this queue too short to be useful in practice, suggesting the thresholds (180 days, 500 impressions) may need loosening in a future iteration.

**Leakage check:** The score is built only from `days_since_last_update`, `impressions_90d`, and `trend_direction` — all observed from the same 90-day trailing window, with no future-window data used anywhere. `trend_direction` is used both inside the rule and as the evaluation label for precision@K; this is flagged here openly, consistent with the flyrank-data skill's label-trap warning, rather than hidden. No FlyRank product decision fields (`health_score`, `priority_score`, `action_type`) were used as inputs to the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.